In [0]:
# Cell 1: Install dependencies
%pip install langgraph>=0.2.0 langchain-openai>=0.2.0 openai databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
# Cell 2: Test if GDPRAgent can be imported and instantiated
import sys
import os
sys.path.append('/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent')

# Set OpenAI API key from Databricks secrets
os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope="openai", key="GDPR_agent")

from gdpr_agent.agent import GDPRAgent

try:
    agent = GDPRAgent()
    print("✅ GDPRAgent instantiated successfully")
    print(f"   Type: {type(agent)}")
except Exception as e:
    print(f"❌ Failed to create GDPRAgent: {e}")
    import traceback
    traceback.print_exc()

In [0]:
# Cell 3: Register model with MLflow wrapper
import mlflow
import os
from gdpr_agent.agent import GDPRAgent

# Ensure OpenAI key is set
os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope="openai", key="GDPR_agent")


class GDPRAgentWrapper(mlflow.pyfunc.PythonModel):
    """MLflow wrapper for GDPRAgent"""
    
    def __init__(self):
        """Initialize without creating agent yet (happens in load_context)"""
        self.agent = None
    
    def load_context(self, context):
        """Called when model is loaded for serving"""
        import os
        # Agent will be initialized when loaded in serving environment
        self.agent = GDPRAgent()
    
    def predict(self, context, model_input):
        """
        Handle inference requests.
        
        Expected input:
        - pandas DataFrame with 'question' column
        - dict with 'question' key
        - list of dicts with 'question' key
        """
        import pandas as pd
        
        # Ensure agent is loaded
        if self.agent is None:
            self.agent = GDPRAgent()
        
        # Handle different input types
        if isinstance(model_input, pd.DataFrame):
            questions = model_input['question'].tolist()
        elif isinstance(model_input, dict):
            questions = [model_input.get('question', '')]
        elif isinstance(model_input, list):
            questions = [q.get('question', '') if isinstance(q, dict) else str(q) for q in model_input]
        else:
            questions = [str(model_input)]
        
        # Process each question
        results = []
        for question in questions:
            try:
                # Call your agent's query method
                response = self.agent.query(question)
                results.append({
                    'answer': response.get('answer', ''),
                    'context': response.get('context', []),
                    'sources': response.get('sources', [])
                })
            except Exception as e:
                results.append({
                    'answer': f"Error: {str(e)}",
                    'context': [],
                    'sources': []
                })
        
        return results


# Register the model
mlflow.set_experiment("/Shared/gdpr-agent-staging")

with mlflow.start_run(run_name="manual_test") as run:
    
    mlflow.log_params({
        "commit_sha": "test123",
        "llm_model": "gpt-4o-mini",
        "deployment_target": "staging"
    })
    
    mlflow.log_metrics({
        "eval_pass_rate": 1.0
    })
    
    # Log the wrapped model
    try:
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=GDPRAgentWrapper(),  # ← Use the wrapper
            registered_model_name="gdpr_agent_staging",
            pip_requirements=[
                "openai>=1.12.0",
                "langgraph>=0.2.0",
                "databricks-vectorsearch",
                "mlflow"
            ]
        )
        print("✅ Model registered successfully!")
        print(f"   Run ID: {run.info.run_id}")
        
    except Exception as e:
        print(f"❌ Failed to register model:")
        print(f"   Error: {e}")
        import traceback
        traceback.print_exc()

In [0]:
# Cell 4a: Restart Python first
dbutils.library.restartPython()

In [0]:
# Test the register_staging.py script
import sys
sys.path.append('/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent')

from deploy.register_staging import register_staging_model

# Test registration
version = register_staging_model(
    commit_sha="test_from_notebook",
    pass_rate=0.95
)

print(f"Registered version: {version}")